# One-time final frozen test evaluation

**Scientific question:** What are the final held-out metrics and paired test-set bootstrap intervals for the four frozen models?
**Configuration:** the immutable four-model registry created by Notebook 09.
**Dataset:** locked 1,927-image clean dataset.
**Split:** the sole 231-sample held-out test, opened only after full preflight.
**Checkpoint:** four validation-selected checkpoints with registry-locked SHA256 values.
**Expected outputs:** per-sample predictions, final metrics/tables, paired bootstrap files, paper figures, and `FINAL_TEST_LOCK.json` under `results/final_test/`.

This notebook performs no training, tuning, checkpoint selection, optimizer creation, backward pass, or `model.train()` call.


## 1. Imports and machine-local data root

Resolve the configured raw-data root without creating any dataset or loader.


In [ ]:
from pathlib import Path
import sys

REPO = Path.cwd().resolve()
while REPO != REPO.parent and not (REPO / "pyproject.toml").is_file():
    REPO = REPO.parent
if not (REPO / "pyproject.toml").is_file():
    raise RuntimeError("Open this notebook from inside the SoilNet repository")
sys.path.insert(0, str(REPO / "src"))

import json
from soilnet.final_sequence import preflight_frozen_registry, run_final_test_once
from soilnet.io import resolve_paths


## 2. Complete frozen-registry preflight

Before test construction, verify exactly four immutable models, exact paths/hashes, split identity/count metadata, validation artifacts, and absence of a completed test lock.


In [ ]:
registry = preflight_frozen_registry()
assert registry["model_count"] == 4 and registry["TEST_OPENED"] == "NO"
print(json.dumps({"preflight": "PASS", "models": [m["experiment_id"] for m in registry["models"]], "test_loader": "NOT_CREATED_YET"}, indent=2))


## 3. Evaluate once and lock

Only after the preceding preflight passes, create one shared test dataset/loader, evaluate all four frozen models under inference mode, persist predictions, bootstrap by paired sample index, plot from saved prediction CSVs, and write the final lock last.


In [ ]:
paths = resolve_paths()
final_lock = run_final_test_once(paths["data_root"])
assert final_lock["TEST_EVALUATED"] == "YES" and final_lock["test_n"] == 231
print(json.dumps(final_lock, indent=2))
